### Chatbot And RAG Evaluation

Retrieval Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by providing them with relevant external knowledge. It has become one of the most widely used approaches for building LLM applications.

This tutorial will show you how to evaluate your RAG applications using LangSmith. You'll learn:

1. How to create test datasets
2. How to run your RAG application on those datasets
3. How to measure your application's performance using different evaluation metrics

#### Overview
A typical RAG evaluation workflow consists of three main steps:

1. Creating a dataset with questions and their expected answers
2. Running your RAG application on those questions
3. Using evaluators to measure how well your application performed, looking at factors like:
 - Answer relevance
 - Answer accuracy
 - Retrieval quality
 
For this tutorial, we'll create and evaluate a bot that answers questions about a few of Lilian Weng's insightful blog posts.

### Chatbot Evaluation

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["MISTRAL_API_KEY"]=os.getenv("MISTRAL_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"

In [2]:
from langsmith import Client

client = Client()

# Define dataset: these are your test cases
dataset_name = "Chatbots Evaluation"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

{'example_ids': ['a5f4837e-b1de-4537-985f-1e0a3b28ff46',
  '7fe2a906-7539-4cb2-b01e-6fab59653e06',
  'd75f944f-8375-4905-81d0-2c54ab85e550',
  'f05f41fd-048f-4f5e-952c-7e596de7cf72',
  'e9052d97-5a02-480c-abff-8f25a85c4a8d'],
 'count': 5,
 'as_of': '2026-05-29T07:02:28.841800477Z'}

### Define Metrics (LLM As A Judge)


In [25]:
import openai
from langsmith import wrappers
from langchain_mistralai import ChatMistralAI

# openai_client = wrappers.wrap_openai(openai.OpenAI())
mistral_client = ChatMistralAI(model="mistral-small-latest", temperature=0)

openai_client = None
if os.getenv("OPENAI_API_KEY"):
    openai_client = wrappers.wrap_openai(openai.OpenAI())

eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = f"""You are grading the following question:
{inputs['question']}
Here is the real answer:
{reference_outputs['answer']}
You are grading the following predicted answer:
{outputs['response']}
Respond with CORRECT or INCORRECT:
Grade:
"""
    response = mistral_client.invoke(
        [
            {"role": "system", "content": eval_instructions},
            {"role": "user", "content": user_content},
        ]
    )
    return response.content.strip() == "CORRECT"


In [26]:
## Concisions- checks whether the actual output is less than 2x the length of the expected result.

def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

### Run Evaluations

In [27]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."

def my_app(question: str, model: str = "mistral-small-latest", instructions: str = default_instructions) -> str:
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": question},
    ]
    if model.startswith("mistral"):
        return mistral_client.invoke(messages).content

    if openai_client is None:
        raise ValueError("OPENAI_API_KEY is not set for OpenAI model usage")

    return openai_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=messages,
    ).choices[0].message.content


In [28]:
### Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])}

In [29]:
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="chatbotchatbot"
)

View the evaluation results for experiment: 'chatbotchatbot-fd49a533' at:
https://smith.langchain.com/o/0193dd3a-3590-4896-8571-0ca3ccb2059b/datasets/9cf79dfd-b40e-436c-a6af-3dee2efb9f73/compare?selectedSessions=ba6131fd-b40d-416f-8222-0b9a2d21bcbc




5it [00:06,  1.33s/it]


In [34]:
### Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"],model="mistral-large-latest")}

In [35]:
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="chatbot by mistral-large-latest"
)

View the evaluation results for experiment: 'chatbot by mistral-large-latest-5951b2bb' at:
https://smith.langchain.com/o/0193dd3a-3590-4896-8571-0ca3ccb2059b/datasets/9cf79dfd-b40e-436c-a6af-3dee2efb9f73/compare?selectedSessions=9cebf284-574e-4bb4-a7bd-02c69005d916




5it [00:06,  1.32s/it]


### Evaluation For RAG

In [40]:
## RAG
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

# List of URLs to load documents from
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

# Load documents from the URLs
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

# Initialize a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)

# Split the documents into chunks
doc_splits = text_splitter.split_documents(docs_list)

class SentenceTransformerEmbeddings:
    def __init__(self, model_name):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts):
        return self.model.encode(texts).tolist()

    def embed_query(self, text):
        return self.model.encode(text).tolist()

# Add the document chunks to the "vector store" using the SentenceTransformer
vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=SentenceTransformerEmbeddings("all-MiniLM-L6-v2")
,
)

# With langchain we can easily turn any vector store into a retrieval component:
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11444.67it/s]


In [16]:
retriever.invoke("what is agents")

[Document(id='618dc4b1-b918-4844-ae2e-86af87f25256', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, 

Failed to multipart ingest runs: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: usage limit monthly_traces of 100 exceeded. Check LangSmith usage configuration settings"}\n')trace=2e2364fd-7c7c-4294-9669-f2c8acc71c76,id=2e2364fd-7c7c-4294-9669-f2c8acc71c76


In [43]:
from langchain.chat_models import init_chat_model
llm=init_chat_model("groq:qwen/qwen3-32b")
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000014DF7D61290>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000014E055C09D0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [44]:
from langsmith import traceable

## Add decorator
@traceable()
def rag_bot(question:str)->dict:
    ## Relevant context
    docs=retriever.invoke(question)
    docs_string = " ".join(doc.page_content for doc in docs)

    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.       Use the following source documents to answer the user's questions.       If you don't know the answer, just say that you don't know.       Use three sentences maximum and keep the answer concise.

Documents:
{docs_string}"""
    
    ## llm invoke

    ai_msg=llm.invoke([
         {"role": "system", "content": instructions},
        {"role": "user", "content": question},

    ])
    return {"answer":ai_msg.content,"documents":docs}



In [45]:
rag_bot("What is agents")

{'answer': '<think>\nOkay, the user asked "What is agents". Let me check the provided documents. The main source is Lilian Weng\'s blog post from June 2023 titled "LLM-powered Autonomous Agents". \n\nLooking through the document, it discusses autonomous agents powered by Large Language Models (LLMs). The blog explains that these agents use LLMs as their core controller, handling tasks like planning, memory, and tool use. Examples given include AutoGPT, GPT-Engineer, and BabyAGI. The agents can break down tasks into subgoals, reflect on past actions, and interact with other agents.\n\nThe user\'s question is pretty general, so I need to define "agents" in this context. The answer should mention that they\'re autonomous systems using LLMs for decision-making, planning, and interaction. Also, include examples and key components like planning, memory, and tool use. Make sure it\'s concise, within three sentences. Avoid technical jargon unless necessary. Check if there\'s any other part of 

### Dataset

In [46]:
from langsmith import Client

client=Client()

# Define the examples for the dataset
examples = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection? "},
        "outputs": {"answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."},
    },
    {
        "inputs": {"question": "What are the types of biases that can arise with few-shot prompting?"},
        "outputs": {"answer": "The biases that can arise with few-shot prompting include (1) Majority label bias, (2) Recency bias, and (3) Common token bias."},
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks?"},
        "outputs": {"answer": "Five types of adversarial attacks are (1) Token manipulation, (2) Gradient based attack, (3) Jailbreak prompting, (4) Human red-teaming, (5) Model red-teaming."},
    }
]

### create the daatset and example in LAngsmith
dataset_name="RAG Test Evaluation"
dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=examples
)



{'example_ids': ['110a1ce7-d69e-4c55-ab02-dedac040fe18',
  'e2bfcc95-e7f9-4acf-b714-f8829b7a874d',
  '4e1fb119-605e-4ae9-bfff-e79e238c0a49'],
 'count': 3,
 'as_of': '2026-05-29T07:46:45.220694973Z'}

### Evaluators or Metrics
1. Correctness: Response vs reference answer
- Goal: Measure "how similar/correct is the RAG chain answer, relative to a ground-truth answer"
- Mode: Requires a ground truth (reference) answer supplied through a dataset
- Evaluator: Use LLM-as-judge to assess answer correctness.

In [ ]:
from typing_extensions import Annotated,TypedDict

## Correctness Output Schema

# Grade output schema
class CorrectnessGrade(TypedDict):
    # Note that the order in the fields are defined is the order in which the model will generate them.
    # It is useful to put explanations before responses because it forces the model to think through
    # its final response before generating it:
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]

## correctness prompt

correctness_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. 
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

from langchain_openai import ChatOpenAI

grader_llm=ChatOpenAI(model="mistral-small-latest",temperature=0,base_url="http://api.mistral.ai/v1").with_structured_output(CorrectnessGrade,
                                                                         method="json_schema",strict=True)
## evaluator
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}"""

    # Run evaluator
    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions}, 
        {"role": "user", "content": answers}
    ])
    return grade["correct"]







### Relevance: Response vs input
The flow is similar to above, but we simply look at the inputs and outputs without needing the reference_outputs. Without a reference answer we can't grade accuracy, but can still grade relevance—as in, did the model address the user's question or not.

In [49]:
# Grade output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "Provide the score on whether the answer addresses the question"]

# Grade prompt
relevance_instructions="""You are a teacher grading a quiz. 

You will be given a QUESTION and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
relevance_llm = ChatOpenAI(model="mistral-small-latest", temperature=0, base_url="http://api.mistral.ai/v1").with_structured_output(RelevanceGrade, method="json_schema", strict=True)

# Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

### Groundedness: Response vs retrieved docs
Another useful way to evaluate responses without needing reference answers is to check if the response is justified by (or "grounded in") the retrieved documents.

In [51]:
# Grade output schema
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[bool, ..., "Provide the score on if the answer hallucinates from the documents"]

# Grade prompt
grounded_instructions = """You are a teacher grading a quiz. 

You will be given FACTS and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. 
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM 
grounded_llm = ChatOpenAI(model="mistral-small-latest", temperature=0, base_url="http://api.mistral.ai/v1").with_structured_output(GroundedGrade, method="json_schema", strict=True)

# Evaluator
def groundedness(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([{"role": "system", "content": grounded_instructions}, {"role": "user", "content": answer}])
    return grade["grounded"]

### Retrieval Relevance: Retrieved docs vs input

In [52]:
# Grade output schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise"]

# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a set of FACTS provided by the student. 

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
retrieval_relevance_llm = ChatOpenAI(model="mistral-small-latest", temperature=0, base_url="http://api.mistral.ai/v1").with_structured_output(RetrievalRelevanceGrade, method="json_schema", strict=True)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"

    # Run evaluator
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

### Run the evaluation

In [54]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness, groundedness, relevance, retrieval_relevance],
    experiment_prefix="rag-doc-relevance",
    metadata={"version": "LCEL context, gpt-4-0125-preview"},
)
# Explore results locally as a dataframe if you have pandas installed
experiment_results.to_pandas()

View the evaluation results for experiment: 'rag-doc-relevance-2bd73369' at:
https://smith.langchain.com/o/0193dd3a-3590-4896-8571-0ca3ccb2059b/datasets/bde4acc9-6a9a-4321-8514-f51d10ca2234/compare?selectedSessions=f4f1f9cb-6a5b-4bed-8e30-fe84e94f3554




0it [00:00, ?it/s]Error running evaluator <DynamicRunEvaluator correctness> on run 019e72b7-fe92-75a0-95ee-0d4e2df55dcf: AuthenticationError("Error code: 401 - {'detail': 'Unauthorized'}")
Traceback (most recent call last):
  File "c:\Users\amrit\Documents\Projects\agent_experiments\.venv\Lib\site-packages\langsmith\evaluation\_runner.py", line 1637, in _run_evaluators
    evaluator_response = evaluator.evaluate_run(  # type: ignore[call-arg]
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\amrit\Documents\Projects\agent_experiments\.venv\Lib\site-packages\langsmith\evaluation\evaluator.py", line 332, in evaluate_run
    result = self.func(
             ^^^^^^^^^^
  File "c:\Users\amrit\Documents\Projects\agent_experiments\.venv\Lib\site-packages\langsmith\run_helpers.py", line 777, in wrapper
    function_result = run_container["context"].run(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\amrit\Documents\Projects\agent_

,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.wrapper,execution_time,example_id,id
0,How does the ReAct agent use self-reflection?,"<think>\nOkay, the user is asking how the ReAc...",[page_content='Self-reflection is a vital aspe...,None,"ReAct integrates reasoning and acting, perform...",None,1.149917,110a1ce7-d69e-4c55-ab02-dedac040fe18,019e72b7-fe92-75a0-95ee-0d4e2df55dcf
1,What are five types of adversarial attacks?,"<think>\nOkay, the user is asking for five typ...",[page_content='Adversarial attacks are inputs ...,None,Five types of adversarial attacks are (1) Toke...,None,0.882546,4e1fb119-605e-4ae9-bfff-e79e238c0a49,019e72b8-0787-7720-b619-f8fe712158c1
2,What are the types of biases that can arise wi...,"<think>\nOkay, the user is asking about the ty...",[page_content='Text: i'll bet the video game i...,None,The biases that can arise with few-shot prompt...,None,0.798048,e2bfcc95-e7f9-4acf-b714-f8829b7a874d,019e72b8-0f14-7241-8436-5328f86e103c
